## HASH `sha2` for IDs
For the Silver OBT, I use IDs like:
```
event_id   = hash of event_name + event_distance_length
result_id  = hash of event_name + event_dates + athlete_id + athlete_performance
```
because `dense_rank()`  is problematic with streaming tables. Deterministic hash ID is better.

HASH
`from pyspark.sql.functions import sha2, concat_ws`
- sha2 creates the same ID every time if the input values are the same.
SO its like:
```
event_name + event_distance_length
→ same event_id every pipeline run
```


## Order inside the Silver script
1. Imports
2. Helper function for snake_case
3. @dp.table(...)
4. Read Bronze marathon table
5. Rename columns
6. Clean event columns
7. Clean athlete columns
8. Read and clean country table
9. Join country table
10. Create event_id and result_id with sha2
11. Drop helper columns
12. Return final Silver OBT dataframe

## Validate OBT created

In [0]:
%sql SHOW TABLES IN marathos_catalog.silver;

In [0]:
%sql
SELECT COUNT(*) AS silver_row_count
FROM marathos_catalog.silver.marathon_results_obt;

## Check country dataset

In [0]:
%sql
SELECT
    athlete_country,
    country_code_iso3,
    country_name,
    COUNT(*) AS row_count
FROM marathos_catalog.silver.marathon_results_obt
GROUP BY athlete_country, country_code_iso3, country_name
ORDER BY row_count DESC;

## check marathos dataset

In [0]:
%sql
SELECT *
FROM marathos_catalog.silver.marathon_results_obt
LIMIT 100;

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT result_id) AS distinct_result_ids,
    COUNT(DISTINCT event_id) AS distinct_event_ids,
    COUNT(DISTINCT athlete_id) AS distinct_athletes
FROM marathos_catalog.silver.marathon_results_obt;

#### Findings: Look for duplicates?

`distinct_result_ids` is slightly lower than `row_count`:

- row_count: 7,340,856
- distinct_result_ids: 7,340,657
- difference: 199

- What is that 199?

In [0]:
%sql
SELECT
    result_id,
    COUNT(*) AS duplicate_count
FROM marathos_catalog.silver.marathon_results_obt
GROUP BY result_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

In [0]:
%sql
    
WITH duplicate_result_ids AS (
    SELECT
        result_id
    FROM marathos_catalog.silver.marathon_results_obt
    GROUP BY result_id
    HAVING COUNT(*) > 1
)

SELECT
    obt.result_id,
    obt.event_id,
    obt.year_of_event,
    obt.event_name,
    obt.event_date_raw,
    obt.event_distance_length,
    obt.athlete_id,
    obt.athlete_country,
    obt.country_name,
    obt.athlete_performance,
    obt.athlete_average_speed,
    obt.athlete_average_speed_kmh
FROM marathos_catalog.silver.marathon_results_obt AS obt
INNER JOIN duplicate_result_ids AS dup
    ON obt.result_id = dup.result_id
ORDER BY
    obt.result_id,
    obt.event_name,
    obt.athlete_id;

In [0]:
%sql
SELECT
    COUNT(*) AS rows_with_duplicate_result_id,
    COUNT(DISTINCT result_id) AS duplicated_result_id_count,
    COUNT(*) - COUNT(DISTINCT result_id) AS extra_duplicate_rows
FROM marathos_catalog.silver.marathon_results_obt
WHERE result_id IN (
    SELECT result_id
    FROM marathos_catalog.silver.marathon_results_obt
    GROUP BY result_id
    HAVING COUNT(*) > 1
);

## Diagnosis Strategy

1. Check if duplicate rows differ in ANY column (including columns not in result_id hash)
2. Compare ALL columns between duplicate rows
3. Identify root cause and fix strategy

In [0]:
%sql
-- Pick one duplicate result_id and compare ALL columns
-- If all columns are identical, these are exact duplicates that should have been caught by dropDuplicates

WITH sample_duplicate AS (
    SELECT result_id
    FROM marathos_catalog.silver.marathon_results_obt
    GROUP BY result_id
    HAVING COUNT(*) > 1
    LIMIT 1
)

SELECT *
FROM marathos_catalog.silver.marathon_results_obt
WHERE result_id = (SELECT result_id FROM sample_duplicate);

In [0]:
%sql
-- For each duplicate result_id, count how many distinct values exist per column
-- If count=1, that column is identical across duplicates
-- If count>1, that column differs

WITH duplicate_groups AS (
    SELECT result_id
    FROM marathos_catalog.silver.marathon_results_obt
    GROUP BY result_id
    HAVING COUNT(*) > 1
),
dup_data AS (
    SELECT obt.*
    FROM marathos_catalog.silver.marathon_results_obt AS obt
    INNER JOIN duplicate_groups ON obt.result_id = duplicate_groups.result_id
)

SELECT
    result_id,
    COUNT(*) AS row_count,
    COUNT(DISTINCT year_of_event) AS distinct_year,
    COUNT(DISTINCT event_dates) AS distinct_event_dates,
    COUNT(DISTINCT event_name) AS distinct_event_name,
    COUNT(DISTINCT event_distance_length) AS distinct_distance_length,
    COUNT(DISTINCT event_number_of_finishers) AS distinct_finishers,
    COUNT(DISTINCT athlete_performance) AS distinct_performance,
    COUNT(DISTINCT athlete_club) AS distinct_club,
    COUNT(DISTINCT athlete_country) AS distinct_country,
    COUNT(DISTINCT athlete_year_of_birth) AS distinct_birth_year,
    COUNT(DISTINCT athlete_gender) AS distinct_gender,
    COUNT(DISTINCT athlete_age_category) AS distinct_age_category,
    COUNT(DISTINCT athlete_id) AS distinct_athlete_id,
    COUNT(DISTINCT event_id) AS distinct_event_id
FROM dup_data
GROUP BY result_id
ORDER BY row_count DESC
LIMIT 20;

In [0]:
%sql
-- Check if duplicates exist in Bronze layer
-- This will tell us if the issue is in the source data or introduced during Silver transformation

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT `Year of event`, `Event name`, `Athlete ID`, `Athlete performance`, `Athlete country`) AS distinct_combinations,
    COUNT(*) - COUNT(DISTINCT `Year of event`, `Event name`, `Athlete ID`, `Athlete performance`, `Athlete country`) AS duplicate_combinations
FROM marathos_catalog.bronze.raw_marathon_results;